# Notebook 4 — Baseline Evaluation and Comparative Analysis
## Phi-4 (Azure) vs Llama 3.1 (Groq) — BIAT Financial Report Generation

**Project:** Small Language Model Evaluation Framework for Structured Financial Analysis
**Author:** [Your name]
**Last updated:** 2026-04
**Case:** test_case_004 — BIAT (Banque Internationale Arabe de Tunisie) 2021–2024

---

### What this notebook does
This notebook loads the model outputs generated in Notebook 2 and evaluates them across
four independent metric layers, producing a ranked comparison of Phi-4 and Llama 3.1
for the task of generating structured financial analysis reports from raw tabular data.

### Evaluation architecture
| Layer | Method | Metrics |
|---|---|---|
| 1 — Deterministic | Reference-based string matching | ROUGE-L, BLEU, ChrF++, BERTScore F1, length ratio |
| 2 — RAGAS | NLI claim decomposition + embeddings | FactualCorrectness, AnswerCorrectness, SemanticSimilarity |
| 3 — Domain | Regex numerical extraction | Financial number match rate (±2% tolerance) |
| 4 — LLM judge | Section-by-section GPT scoring | Faithfulness, Numerical accuracy, Trend accuracy, Reasoning, Completeness, Hallucination |

### Key design decisions
- Each model is evaluated across **5 temperature runs** (0.0, 0.1, 0.3, 0.5, 0.7) to measure output stability
- RAGAS metrics run **section-by-section** (not holistically) for diagnostic precision
- A **two-judge ensemble** averages scores from two independent LLM judges to reduce single-judge bias
- A **judge cache** avoids redundant API calls on re-runs
- All results are logged to **MLflow** for lineage and reproducibility

### Interpretation note
This is a **single-case directional benchmark** (n=1 financial report, 5 temperature runs).
Score differences indicate direction of advantage but are not statistically significant.
Scale to n≥20 cases before drawing deployment conclusions.

---

### Table of contents
1. [Imports and configuration](#imports)
2. [Load baseline outputs](#load)
3. [Section extraction and diagnostics](#sections)
4. [Layer 1 — Deterministic metrics](#layer1)
5. [Layer 4 — LLM judge (section-by-section)](#judge)
6. [Layer 3 — Domain numerical accuracy](#layer3)
7. [Layer 2 — RAGAS metrics](#layer2)
8. [Variance summary across temperature runs](#variance)
9. [Operational metrics (latency, tokens, cost)](#ops)
10. [Final comparison table and radar chart](#comparison)
11. [MLflow logging and artifact export](#mlflow)

In [39]:
from __future__ import annotations

import os
import json
import hashlib
import re
from pydantic import BaseModel
from pathlib import Path

import numpy as np
import pandas as pd
import mlflow
import openai

from dotenv import load_dotenv
from openai import OpenAI

from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from bert_score import score as bertscore_score
from evaluate import load as load_metric
from ragas import evaluate as ragas_evaluate, EvaluationDataset, SingleTurnSample
from ragas.metrics import FactualCorrectness, AnswerCorrectness, SemanticSimilarity
from ragas.llms import LangchainLLMWrapper
from langchain_openai import ChatOpenAI

import plotly.graph_objects as go

C:\Users\BRHN\AppData\Local\Temp\ipykernel_300\744295277.py:23: DeprecationWarning: Importing FactualCorrectness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import FactualCorrectness
  from ragas.metrics import FactualCorrectness, AnswerCorrectness, SemanticSimilarity
C:\Users\BRHN\AppData\Local\Temp\ipykernel_300\744295277.py:23: DeprecationWarning: Importing AnswerCorrectness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerCorrectness
  from ragas.metrics import FactualCorrectness, AnswerCorrectness, SemanticSimilarity
C:\Users\BRHN\AppData\Local\Temp\ipykernel_300\744295277.py:23: DeprecationWarning: Importing SemanticSimilarity from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.met

In [40]:
# Project paths
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

ENV_PATH = PROJECT_ROOT / ".env"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
REPORTS_DIR = OUTPUTS_DIR / "reports"
FIGURES_DIR = OUTPUTS_DIR / "figures"
CACHE_DIR = OUTPUTS_DIR / "cache"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

load_dotenv(ENV_PATH, override=True)

# MLflow configuration
MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI")
MLFLOW_EXPERIMENT_NAME = os.getenv("MLFLOW_EXPERIMENT_NAME")
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

print("Environment ready.")
print("MLflow version:", mlflow.__version__)
print("OpenAI version:", openai.__version__)

Environment ready.
MLflow version: 3.10.1
OpenAI version: 2.29.0


---
## ⚙️ Configuration

Sets all evaluation flags and metric weights in one place.
Change `FORCE_RERUN_JUDGE = True` to bypass the judge cache and re-score everything fresh.
Change `RUN_RAGAS = False` to skip the RAGAS layer (saves ~40k tokens on Groq).

**Score weights** define how the final 0–1 score is computed:
the LLM judge carries the most weight (38%) because factual accuracy and reasoning
matter more than lexical overlap for financial report evaluation.

In [41]:
mlflow.end_run()

In [42]:
CASE_ID = "test_case_004"
nb4_run = mlflow.start_run(run_name=f"nb4_evaluation_{CASE_ID}")
eval_run_id = nb4_run.info.run_id

FORCE_RERUN_JUDGE = False
RUN_RAGAS = True

SCORE_WEIGHTS = {
    "judge_quality_score": 0.38,
    "semantic_score": 0.19,
    "structure_score": 0.1425,
    "safety_score": 0.1425,
    "efficiency_score": 0.095,
    "chrf": 0.05,
}

N_RUNS = 5            # number of generation runs per model for variance measurement
JUDGE_TEMPERATURE = 0.0
RAGAS_ENABLED = True
SECOND_JUDGE_MODEL = os.getenv("GROQ_JUDGE_MODEL_2")
PRIMARY_JUDGE_MODEL = os.getenv("GROQ_JUDGE_MODEL")
if not SECOND_JUDGE_MODEL:
    print("WARNING: Set GROQ_JUDGE_MODEL_2 to use a distinct second judge model.")
elif SECOND_JUDGE_MODEL == PRIMARY_JUDGE_MODEL:
    print("WARNING: GROQ_JUDGE_MODEL_2 matches GROQ_JUDGE_MODEL; second judge is not independent.")

if not np.isclose(sum(SCORE_WEIGHTS.values()), 1.0):
    raise ValueError(f"SCORE_WEIGHTS must sum to 1.0, got {sum(SCORE_WEIGHTS.values()):.3f}")


---
## 📂 Load baseline outputs

Loads the CSV produced by Notebook 2 which contains one row per model per temperature run.
With 2 models × 5 temperatures = **10 rows**.
Required columns are validated before proceeding — missing columns raise immediately.

In [43]:
OUTPUTS_CSV_PATH = OUTPUTS_DIR / f"baseline_outputs_{CASE_ID}.csv"
if not OUTPUTS_CSV_PATH.exists():
    raise FileNotFoundError(f"Missing outputs file: {OUTPUTS_CSV_PATH}")

outputs_df = pd.read_csv(OUTPUTS_CSV_PATH)

# Confirm that required columns exist
required_cols = [
    "case_id",
    "model_label",
    "provider",
    "latency_sec",
    "prompt_tokens",
    "completion_tokens",
    "total_tokens",
    "output_text",
    "reference_output",
]
missing = [c for c in required_cols if c not in outputs_df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

# Convert latency to milliseconds if desired
outputs_df["latency_ms"] = outputs_df["latency_sec"] * 1000

outputs_df.head()

,case_id,provider,model_label,model_name,prompt_uri,prompt_name,prompt_version,temperature,max_tokens,latency_sec,...,has_balance_sheet_structure_and_liquidity,has_capital_adequacy,has_key_risks_and_watch_points,has_conclusion,structure_score,missing_sections_count,missing_sections,parsed_sections_json,reference_sections_json,latency_ms
0,test_case_004,azure_openai_compatible,Phi-4,Phi-4-experimentation-investment-nextgen,prompts:/financial_analysis_baseline/11,financial_analysis_baseline,11,0.0,1200,22.6712,...,1,1,1,1,1.0,0,[],"{""Executive Summary"": ""Banque Internationale A...","{""Executive Summary"": ""Over the 2021-2024 peri...",22671.2
1,test_case_004,groq,Llama 3.1,llama-3.1-8b-instant,prompts:/financial_analysis_baseline/11,financial_analysis_baseline,11,0.0,1200,1.3880,...,1,1,1,1,1.0,0,[],"{""Executive Summary"": ""Banque Internationale A...","{""Executive Summary"": ""Over the 2021-2024 peri...",1388.0
2,test_case_004,azure_openai_compatible,Phi-4,Phi-4-experimentation-investment-nextgen,prompts:/financial_analysis_baseline/11,financial_analysis_baseline,11,0.1,1200,27.4613,...,1,1,1,1,1.0,0,[],"{""Executive Summary"": ""Banque Internationale A...","{""Executive Summary"": ""Over the 2021-2024 peri...",27461.3
3,test_case_004,groq,Llama 3.1,llama-3.1-8b-instant,prompts:/financial_analysis_baseline/11,financial_analysis_baseline,11,0.1,1200,5.0886,...,1,1,1,1,1.0,0,[],"{""Executive Summary"": ""Banque Internationale A...","{""Executive Summary"": ""Over the 2021-2024 peri...",5088.6
4,test_case_004,azure_openai_compatible,Phi-4,Phi-4-experimentation-investment-nextgen,prompts:/financial_analysis_baseline/11,financial_analysis_baseline,11,0.3,1200,23.8216,...,1,1,1,1,1.0,0,[],"{""Executive Summary"": ""Banque Internationale A...","{""Executive Summary"": ""Over the 2021-2024 peri...",23821.6


---
## 🔍 Section extraction

The prompt instructs both models to produce a report with **8 fixed sections** in a fixed order.
This block defines the canonical section names, allowed aliases (to handle minor wording
variations), and the regex-based extractor that splits each report into its sections.

Section extraction is critical: the LLM judge, RAGAS section-by-section metrics, and the
structure score all depend on cleanly parsed sections. If a model merges or renames sections,
it is detected here and penalised in the structure score.

In [44]:
EXPECTED_SECTIONS = [
    "Executive Summary",
    "Profitability and Operational Efficiency",
    "Revenue Dynamics",
    "Asset Quality and Risk Profile",
    "Balance Sheet Structure and Liquidity",
    "Capital Adequacy",
    "Key Risks and Watch Points",
    "Conclusion",
]

In [45]:
SECTION_ALIASES = {
    "Executive Summary": ["Executive Summary", "Summary", "Overview"],
    "Profitability and Operational Efficiency": [
        "Profitability and Operational Efficiency",
        "Profitability and Efficiency",
        "Profitability & Efficiency",
    ],
    "Revenue Dynamics": ["Revenue Dynamics"],
    "Asset Quality and Risk Profile": [
        "Asset Quality and Risk Profile",
        "Risk Profile",
    ],
    "Balance Sheet Structure and Liquidity": [
        "Balance Sheet Structure and Liquidity",
        "Liquidity & Balance Sheet",
        "Liquidity and Balance Sheet",
    ],
    "Capital Adequacy": ["Capital Adequacy"],
    "Key Risks and Watch Points": [
        "Key Risks and Watch Points",
        "Risks & Watch Points",
        "Risks and Watch Points",
    ],
    "Conclusion": ["Conclusion"],
}

def normalize_report_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = text.strip()
    text = re.sub(r"\r\n?", "\n", text)
    text = re.sub(r"[–—−]", "-", text)
    # Remove bold markdown formatting (**text** -> text)
    text = re.sub(r"\*\*(.+?)\*\*", r"\1", text)
    return text

def extract_sections_rule_based(text: str) -> dict[str, str]:
    text = normalize_report_text(text)
    results = {name: "" for name in EXPECTED_SECTIONS}
    matches = []

    for canonical, aliases in SECTION_ALIASES.items():
        for alias in aliases:
            pattern = rf"(?im)^[ \t]*(?:#+[ \t]*)?{re.escape(alias)}[ \t]*:?[ \t]*$"
            for m in re.finditer(pattern, text):
                matches.append((m.start(), m.end(), canonical, alias))

    matches.sort(key=lambda x: x[0])

    deduped = []
    seen = set()
    for item in matches:
        key = (item[0], item[2])
        if key not in seen:
            deduped.append(item)
            seen.add(key)

    for i, (_, end_pos, canonical, _) in enumerate(deduped):
        next_start = deduped[i + 1][0] if i + 1 < len(deduped) else len(text)
        results[canonical] = text[end_pos:next_start].strip()

    return results

In [46]:
def extract_sections(text: str) -> dict[str, str]:
    return extract_sections_rule_based(text)

## Normalize report text


In [47]:
# Re-normalize outputs with the UPDATED normalize_report_text function
# (This ensures bold markdown is properly removed)
outputs_df["output_text_normalized"] = outputs_df["output_text"].apply(normalize_report_text)
outputs_df["reference_output_normalized"] = outputs_df["reference_output"].apply(normalize_report_text)

print("✅ Re-normalized with updated function that removes bold markdown")

✅ Re-normalized with updated function that removes bold markdown


## Section extraction diagnostics


In [48]:
section_diagnostics = []

for _, row in outputs_df.iterrows():
    for text_col, label in [
        ("reference_output", "reference"),
        ("output_text", "model_output"),
    ]:
        sections = extract_sections(row[text_col])
        present_count = sum(bool(text.strip()) for text in sections.values())
        missing = [name for name, text in sections.items() if not text.strip()]
        section_diagnostics.append({
            "model_label": row["model_label"],
            "text_type": label,
            "sections_found": present_count,
            "sections_expected": len(EXPECTED_SECTIONS),
            "missing_sections": ", ".join(missing),
        })

section_diagnostics_df = pd.DataFrame(section_diagnostics)
display(section_diagnostics_df)


,model_label,text_type,sections_found,sections_expected,missing_sections
0,Phi-4,reference,8,8,
1,Phi-4,model_output,8,8,
2,Llama 3.1,reference,8,8,
3,Llama 3.1,model_output,8,8,
4,Phi-4,reference,8,8,
5,Phi-4,model_output,8,8,
6,Llama 3.1,reference,8,8,
7,Llama 3.1,model_output,8,8,
8,Phi-4,reference,8,8,
9,Phi-4,model_output,8,8,


## Metric helpers

In [49]:
# Rouge, ChrF++, and BLEU helper objects
rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
_chrf = load_metric("chrf")
smooth = SmoothingFunction().method1

def compute_rouge_l(reference: str, prediction: str) -> float:
    return rouge.score(reference, prediction)["rougeL"].fmeasure

def compute_chrf(reference: str, prediction: str) -> float:
    """ChrF++ score — better than BLEU for long-form structured text."""
    if not reference.strip() or not prediction.strip():
        return 0.0
    result = _chrf.compute(
        predictions=[prediction],
        references=[reference],
        word_order=2,        # ChrF++ variant
        beta=2,              # recall-weighted
    )
    return round(float(result["score"]) / 100.0, 4)   # normalise 0-100 → 0-1

def compute_bleu(reference: str, prediction: str) -> float:
    reference_tokens = reference.split()
    prediction_tokens = prediction.split()
    if not reference_tokens or not prediction_tokens:
        return 0.0
    return sentence_bleu([reference_tokens], prediction_tokens, smoothing_function=smooth)

def compute_output_length(text: str) -> int:
    return len(text.split()) if isinstance(text, str) else 0

def normalized_length_score(prediction: str, reference: str) -> float:
    pred_len = compute_output_length(prediction)
    ref_len = compute_output_length(reference)
    if ref_len == 0:
        return 0.0
    score = 1 - abs(pred_len - ref_len) / ref_len
    return float(max(0.0, min(1.0, score)))



def invert_minmax(series: pd.Series) -> pd.Series:
    series = series.astype(float)
    min_v, max_v = series.min(), series.max()
    if max_v == min_v:
        return pd.Series([1.0] * len(series), index=series.index)
    normalized = (series - min_v) / (max_v - min_v)
    inverted = 1 - normalized
    return inverted.clip(0, 1)

In [50]:
def structure_components(text: str) -> dict:
    sections = extract_sections_rule_based(text)
    present = {k: int(bool(v.strip())) for k, v in sections.items()}
    missing_sections = [k for k, v in sections.items() if not v.strip()]
    structure_score = sum(present.values()) / len(EXPECTED_SECTIONS)

    return {
        "structure_score": structure_score,
        "missing_sections_count": len(missing_sections),
        "missing_sections": ", ".join(missing_sections) if missing_sections else "",
        **{f"has_{k.lower().replace(' ', '_').replace('&', 'and').replace('-', '_')}": v for k, v in present.items()},
    }

---
## 📏 Layer 1 — Deterministic metrics

Fast, reproducible, no API calls. Run on all 10 rows (2 models × 5 temperature runs).

| Metric | What it measures | Range |
|---|---|---|
| **ROUGE-L** | Longest common subsequence overlap with reference | 0–1 |
| **BLEU** | N-gram precision (kept as secondary regression signal) | 0–1 |
| **ChrF++** | Character n-gram F-score — better than BLEU for long-form text | 0–1 |
| **BERTScore F1** | Semantic similarity via BERT embeddings | 0–1 |
| **Length score** | Penalises outputs much shorter or longer than the reference | 0–1 |
| **Structure score** | Fraction of the 8 required sections present | 0–1 |

These metrics serve as fast regression checks. A drop in ROUGE-L between prompt versions
signals structural degradation. ChrF++ is the primary lexical metric — it handles
paraphrasing and morphological variation better than BLEU.

In [51]:
outputs_df["rougeL"] = outputs_df.apply(
    lambda row: compute_rouge_l(row["reference_output_normalized"], row["output_text_normalized"]),
    axis=1,
)

outputs_df["bleu"] = outputs_df.apply(
    lambda row: compute_bleu(row["reference_output_normalized"], row["output_text_normalized"]),
    axis=1,
)
outputs_df["chrf"] = outputs_df.apply(
    lambda row: compute_chrf(
        row["reference_output_normalized"], row["output_text_normalized"]
    ),
    axis=1,
)

outputs_df["length_score"] = outputs_df.apply(
    lambda row: normalized_length_score(row["output_text_normalized"], row["reference_output_normalized"]),
    axis=1,
)

structure_columns = [
    "structure_score",
    "missing_sections_count",
    "missing_sections",
] + [col for col in outputs_df.columns if col.startswith("has_")]
outputs_df = outputs_df.drop(columns=[col for col in structure_columns if col in outputs_df.columns])

structure_df = outputs_df["output_text_normalized"].apply(structure_components).apply(pd.Series)
outputs_df = pd.concat([outputs_df, structure_df], axis=1)

P, R, F1 = bertscore_score(
    outputs_df["output_text_normalized"].tolist(),
    outputs_df["reference_output_normalized"].tolist(),
    lang="en",
    verbose=False,
)
outputs_df["bert_precision"] = P.detach().cpu().numpy()
outputs_df["bert_recall"] = R.detach().cpu().numpy()
outputs_df["bert_f1"] = F1.detach().cpu().numpy()

display(outputs_df[["model_label", "rougeL", "bleu", "chrf", "bert_f1", "length_score", "structure_score"]])


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


,model_label,rougeL,bleu,chrf,bert_f1,length_score,structure_score
0,Phi-4,0.290512,0.108059,0.4361,0.889214,0.732116,1.0
1,Llama 3.1,0.288541,0.076551,0.3765,0.876069,0.757991,1.0
2,Phi-4,0.296825,0.128460,0.4657,0.890521,0.838661,1.0
3,Llama 3.1,0.277338,0.118017,0.3884,0.887078,0.824962,1.0
4,Phi-4,0.297483,0.134777,0.4702,0.889592,0.896499,1.0
5,Llama 3.1,0.258741,0.051041,0.3132,0.868028,0.681887,1.0
6,Phi-4,0.285930,0.108061,0.4776,0.891150,0.920852,1.0
7,Llama 3.1,0.270091,0.104825,0.3470,0.885300,0.744292,1.0
8,Phi-4,0.307105,0.123417,0.4855,0.889960,0.904110,1.0
9,Llama 3.1,0.280423,0.074546,0.3489,0.881182,0.649924,1.0


## Judge configuration


In [52]:
# Groq judge settings from .env
JUDGE_API_KEY = os.getenv("JUDGE_API_KEY")      # Groq API key
JUDGE_BASE_URL = os.getenv("JUDGE_BASE_URL")    # e.g. https://api.groq.com/openai/v1
GROQ_JUDGE_MODEL = os.getenv("GROQ_JUDGE_MODEL")  # e.g. "openai/gpt-oss-20b"
GROQ_JUDGE_MODEL_2 = os.getenv("GROQ_JUDGE_MODEL_2")  # e.g. "openai/gpt-oss-20b"

# Validate env
required_env = {
    "JUDGE_API_KEY": JUDGE_API_KEY,
    "JUDGE_BASE_URL": JUDGE_BASE_URL,
    "GROQ_JUDGE_MODEL": GROQ_JUDGE_MODEL,
    "GROQ_JUDGE_MODEL_2": GROQ_JUDGE_MODEL_2,
    "MLFLOW_TRACKING_URI": MLFLOW_TRACKING_URI,
    "MLFLOW_EXPERIMENT_NAME": MLFLOW_EXPERIMENT_NAME,
}
print("Environment validation:")
for key, val in required_env.items():
    print(f" - {key}: {'OK' if val else 'MISSING'}")
if any(v is None or v == "" for v in required_env.values()):
    raise EnvironmentError("Missing required environment variables.")

Environment validation:
 - JUDGE_API_KEY: OK
 - JUDGE_BASE_URL: OK
 - GROQ_JUDGE_MODEL: OK
 - GROQ_JUDGE_MODEL_2: OK
 - MLFLOW_TRACKING_URI: OK
 - MLFLOW_EXPERIMENT_NAME: OK


## Judge function


In [53]:
client = OpenAI(
    base_url=JUDGE_BASE_URL,
    api_key=JUDGE_API_KEY,
)
judge_model_name = GROQ_JUDGE_MODEL

client_judge2 = OpenAI(
    base_url=JUDGE_BASE_URL,
    api_key=JUDGE_API_KEY,
)
judge2_model_name = SECOND_JUDGE_MODEL   # defined in TASK 0
use_ensemble_judge = bool(judge2_model_name) and (judge2_model_name != GROQ_JUDGE_MODEL)
print(f"Ensemble judge enabled: {use_ensemble_judge}")
print(f"Judge 1: {GROQ_JUDGE_MODEL} | Judge 2: {judge2_model_name}")


def evaluate_with_judge(model_output: str, reference_output: str) -> dict:
    model_sections = extract_sections(model_output)
    ref_sections = extract_sections(reference_output)

    sections_prompt = []
    for name in EXPECTED_SECTIONS:
        model_text = model_sections.get(name, "")
        ref_text = ref_sections.get(name, "")
        sections_prompt.append(
            f"### {name}\n"
            f"Model Output:\n\"\"\"\n{model_text}\n\"\"\"\n"
            f"Ground Truth:\n\"\"\"\n{ref_text}\n\"\"\"\n"
        )
    sections_prompt = "\n".join(sections_prompt)

    json_template = {
        section: {
            "faithfulness": 0,
            "numerical_accuracy": 0,
            "trend_accuracy": 0,
            "reasoning_quality": 0,
            "completeness": 0,
            "hallucination": 0,
            "justification": ""
        }
        for section in EXPECTED_SECTIONS
    }

    system_instruction = (
        "You are a strict evaluation judge for financial reports. "
        "Return ONLY a valid JSON object. "
        "Do not include markdown fences, comments, explanations, or extra text outside JSON. "
        "Each score must be an integer from 0 to 10. "
        "Penalize hallucinations heavily. "
        "If a section is missing or merged, assign a low completeness score and explain why in justification."
    )

    user_prompt = (
        "Evaluate the model output section by section against the ground truth.\n\n"
        "Criteria for each section:\n"
        "- faithfulness\n"
        "- numerical_accuracy\n"
        "- trend_accuracy\n"
        "- reasoning_quality\n"
        "- completeness\n"
        "- hallucination\n\n"
        "Use this exact JSON structure and the same section names:\n"
        f"{json.dumps(json_template, indent=2)}\n\n"
        f"{sections_prompt}"
    )

    response = client.chat.completions.create(
        model=judge_model_name,
        messages=[
            {"role": "system", "content": system_instruction},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.0,
        max_tokens=3000,
        response_format={"type": "json_object"},
    )

    raw = response.choices[0].message.content.strip()

    raw = re.sub(r"^```json\s*", "", raw)
    raw = re.sub(r"^```", "", raw)
    raw = re.sub(r"```$", "", raw).strip()

    try:
        parsed = json.loads(raw)
        return parsed
    except Exception:
        return {
            "error": "Failed to parse JSON",
            "raw_response": raw,
        }

def _average_judge_values(value1, value2):
    if isinstance(value1, (int, float)) and isinstance(value2, (int, float)):
        return round((value1 + value2) / 2, 4)
    if isinstance(value1, dict) and isinstance(value2, dict):
        merged = {}
        for key in set(value1) | set(value2):
            if key in value1 and key in value2:
                merged[key] = _average_judge_values(value1[key], value2[key])
            else:
                merged[key] = value1.get(key, value2.get(key))
        return merged
    return value1


def _collect_numeric_diffs(value1, value2) -> list[float]:
    if isinstance(value1, (int, float)) and isinstance(value2, (int, float)):
        return [abs(float(value1) - float(value2)) / 10.0]
    if isinstance(value1, dict) and isinstance(value2, dict):
        diffs = []
        for key in set(value1) & set(value2):
            diffs.extend(_collect_numeric_diffs(value1[key], value2[key]))
        return diffs
    return []


def _attach_judge_copies(merged: dict, result1: dict, result2: dict) -> None:
    for section, scores in result1.items():
        if not isinstance(scores, dict) or not isinstance(result2.get(section), dict):
            continue
        merged_section = merged.setdefault(section, {})
        for key, value1 in scores.items():
            value2 = result2[section].get(key)
            if isinstance(value1, (int, float)) and isinstance(value2, (int, float)):
                merged_section[f"{key}_j1"] = value1
                merged_section[f"{key}_j2"] = value2


def evaluate_with_judge_ensemble(model_output: str, reference_output: str) -> dict:
    """
    Runs evaluate_with_judge() with two models and returns averaged scores.
    Falls back to single judge if both models are the same.
    """
    global client, judge_model_name

    result1 = evaluate_with_judge(model_output, reference_output)

    if not use_ensemble_judge:
        result1["judge_agreement"] = 1.0   # trivially agree with itself
        return result1

    # Temporarily swap the global client to judge 2
    # (evaluate_with_judge uses the module-level `client` variable)
    _orig_client, _orig_model = client, judge_model_name
    client, judge_model_name = client_judge2, judge2_model_name
    try:
        result2 = evaluate_with_judge(model_output, reference_output)
    finally:
        client, judge_model_name = _orig_client, _orig_model   # restore

    if not isinstance(result1, dict) or not isinstance(result2, dict):
        return result1
    if "error" in result1 or "error" in result2:
        result1["judge2_result"] = result2
        result1["judge_agreement"] = np.nan
        return result1

    merged = _average_judge_values(result1, result2)
    _attach_judge_copies(merged, result1, result2)

    # Compute simple agreement score: mean absolute diff across section scores
    diffs = _collect_numeric_diffs(result1, result2)
    if diffs:
        merged["judge_agreement"] = round(1.0 - (sum(diffs) / len(diffs)), 4)

    return merged



Ensemble judge enabled: True
Judge 1: openai/gpt-oss-20b | Judge 2: meta-llama/llama-4-scout-17b-16e-instruct


---
## 🧑‍⚖️ Layer 4 — LLM judge (section-by-section)

The judge evaluates each of the 8 sections independently against the golden reference.
It scores 6 criteria per section on a 0–10 scale:

| Criterion | What it penalises |
|---|---|
| **Faithfulness** | Claims not supported by the input financial data |
| **Numerical accuracy** | Wrong figures, ratios, or percentages |
| **Trend accuracy** | Incorrect direction (e.g. saying NPL improved when it worsened) |
| **Reasoning quality** | Superficial or circular analysis |
| **Completeness** | Missing key observations from the section |
| **Hallucination** | Invented facts with no basis in the input — heavily penalised |

A **two-judge ensemble** is used when `GROQ_JUDGE_MODEL_2` differs from `GROQ_JUDGE_MODEL`.
Scores are averaged across both judges. Inter-judge agreement is logged as a meta-metric
— values below 0.70 indicate the judge prompt may need calibration.

A **cache** stores results keyed by output hash + judge model. Re-running the notebook
without changing outputs costs zero additional API calls.

In [54]:
JUDGE_CACHE_PATH = CACHE_DIR / f"judge_eval_{CASE_ID}_{judge_model_name.replace('/', '_')}.json"


def _text_hash(*values: object) -> str:
    payload = "\n".join("" if value is None else str(value) for value in values)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()[:16]


def _judge_cache_key(row: pd.Series) -> str:
    return "|".join([
        str(row["case_id"]),
        str(row["provider"]),
        str(row["model_label"]),
        str(row.get("model_name", "")),
        str(judge_model_name),
        str(judge2_model_name),
        str(use_ensemble_judge),
        _text_hash(row["output_text"], row["reference_output"]),
    ])


def load_judge_cache(path: Path) -> dict:
    if not path.exists():
        return {}
    return json.loads(path.read_text(encoding="utf-8"))


def save_judge_cache(path: Path, cache: dict) -> None:
    path.write_text(json.dumps(cache, ensure_ascii=False, indent=2), encoding="utf-8")


judge_cache = {} if FORCE_RERUN_JUDGE else load_judge_cache(JUDGE_CACHE_PATH)
cache_hits = 0
cache_misses = 0
llm_evals = []

for _, row in outputs_df.iterrows():
    cache_key = _judge_cache_key(row)
    if cache_key in judge_cache:
        llm_evals.append(judge_cache[cache_key])
        cache_hits += 1
        continue

    result = evaluate_with_judge_ensemble(row["output_text"], row["reference_output"])
    judge_cache[cache_key] = result
    llm_evals.append(result)
    cache_misses += 1

outputs_df["llm_eval"] = llm_evals
outputs_df["judge_agreement"] = [
    result.get("judge_agreement", np.nan) if isinstance(result, dict) else np.nan
    for result in llm_evals
]
save_judge_cache(JUDGE_CACHE_PATH, judge_cache)

print(f"Judge cache: {JUDGE_CACHE_PATH}")
print(f"Cache hits: {cache_hits}")
print(f"Cache misses: {cache_misses}")

outputs_df[["model_label", "llm_eval"]]


Judge cache: c:\Users\BRHN\Desktop\SLM-evals\outputs\cache\judge_eval_test_case_004_openai_gpt-oss-20b.json
Cache hits: 10
Cache misses: 0


,model_label,llm_eval
0,Phi-4,{'Balance Sheet Structure and Liquidity': {'fa...
1,Llama 3.1,{'Balance Sheet Structure and Liquidity': {'fa...
2,Phi-4,{'Balance Sheet Structure and Liquidity': {'fa...
3,Llama 3.1,{'Balance Sheet Structure and Liquidity': {'fa...
4,Phi-4,{'Balance Sheet Structure and Liquidity': {'fa...
5,Llama 3.1,{'Balance Sheet Structure and Liquidity': {'fa...
6,Phi-4,{'Balance Sheet Structure and Liquidity': {'fa...
7,Llama 3.1,{'Balance Sheet Structure and Liquidity': {'fa...
8,Phi-4,{'Balance Sheet Structure and Liquidity': {'fa...
9,Llama 3.1,{'Balance Sheet Structure and Liquidity': {'fa...


## Flatten judge results


In [55]:
judge_metrics = [
    "faithfulness",
    "numerical_accuracy",
    "trend_accuracy",
    "reasoning_quality",
    "completeness",
    "hallucination",
]

rows = []

for output_row_id, row in outputs_df.iterrows():
    eval_dict = row["llm_eval"]
    if not isinstance(eval_dict, dict) or "error" in eval_dict:
        rows.append({
            "output_row_id": output_row_id,
            "run_seed": row.get("run_seed", np.nan),
            "model_label": row["model_label"],
            "section": "__evaluation_error__",
            **{m: np.nan for m in judge_metrics},
            "justification": eval_dict.get("raw_response", eval_dict.get("error", "Unknown judge error"))
            if isinstance(eval_dict, dict) else "Invalid judge response",
        })
        continue

    for section in EXPECTED_SECTIONS:
        scores = eval_dict.get(section, {})
        rows.append({
            "output_row_id": output_row_id,
            "run_seed": row.get("run_seed", np.nan),
            "model_label": row["model_label"],
            "section": section,
            **{m: scores.get(m) for m in judge_metrics},
            "justification": scores.get("justification", ""),
        })

judge_df = pd.DataFrame(rows)

for metric in judge_metrics:
    judge_df[metric] = pd.to_numeric(judge_df[metric], errors="coerce")

judge_df["quality_score"] = judge_df[
    ["faithfulness", "numerical_accuracy", "trend_accuracy", "reasoning_quality", "completeness"]
].mean(axis=1) / 10
judge_df["safety_score"] = 1 - (judge_df["hallucination"] / 10)
judge_df["section_score"] = (0.80 * judge_df["quality_score"] + 0.20 * judge_df["safety_score"]).clip(0, 1)

per_output_judge_scores = (
    judge_df[judge_df["section"].isin(EXPECTED_SECTIONS)]
    .groupby("output_row_id")[judge_metrics + ["quality_score", "safety_score", "section_score"]]
    .mean()
)
for metric in judge_metrics:
    outputs_df[metric] = per_output_judge_scores[metric]
outputs_df["judge_quality_score"] = per_output_judge_scores["quality_score"]
outputs_df["safety_score"] = per_output_judge_scores["safety_score"]
outputs_df["judge_section_score"] = per_output_judge_scores["section_score"]

display(judge_df)


,output_row_id,run_seed,model_label,section,faithfulness,numerical_accuracy,trend_accuracy,reasoning_quality,completeness,hallucination,justification,quality_score,safety_score,section_score
0,0,0,Phi-4,Executive Summary,7.0,9.0,8.0,7.0,6.0,5.5,The summary captures the overall growth and im...,0.74,0.45,0.682
1,0,0,Phi-4,Profitability and Operational Efficiency,7.5,7.5,8.0,7.0,6.5,5.0,The model correctly states the cost‑to‑income ...,0.73,0.50,0.684
2,0,0,Phi-4,Revenue Dynamics,8.0,8.5,8.5,6.5,6.0,5.5,Growth rates match the ground truth’s descript...,0.75,0.45,0.690
3,0,0,Phi-4,Asset Quality and Risk Profile,7.5,8.0,7.5,6.5,5.5,6.0,The NPL trend and cost‑of‑risk decline are cor...,0.70,0.40,0.640
4,0,0,Phi-4,Balance Sheet Structure and Liquidity,8.5,8.5,8.5,7.0,6.5,5.0,The LDR and LCR trends are consistent with the...,0.78,0.50,0.724
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,9,4,Llama 3.1,Asset Quality and Risk Profile,6.0,6.0,6.0,4.5,3.5,1.5,The model captures the general stability of NP...,0.52,0.85,0.586
76,9,4,Llama 3.1,Balance Sheet Structure and Liquidity,5.0,4.0,4.0,4.5,4.0,3.5,The asset figures are not provided in the grou...,0.43,0.65,0.474
77,9,4,Llama 3.1,Capital Adequacy,8.0,9.0,9.0,4.5,4.5,1.5,The CET1 trend is accurately reported and matc...,0.70,0.85,0.730
78,9,4,Llama 3.1,Key Risks and Watch Points,0.0,0.0,0.0,0.0,0.0,5.0,"The model provides no risk information, wherea...",0.00,0.50,0.100


## Section-level judge insights


In [56]:
section_best_style = "background-color: #fff2cc; color: #7a4f00; font-weight: 700;"

judge_display_columns = {
    "model_label": "Model",
    "section": "Section",
    "section_score": "Section Score",
    "quality_score": "Quality",
    "safety_score": "Safety",
    "faithfulness": "Faithfulness",
    "numerical_accuracy": "Numerical Accuracy",
    "trend_accuracy": "Trend Accuracy",
    "reasoning_quality": "Reasoning",
    "completeness": "Completeness",
    "hallucination": "Hallucination",
    "justification": "Judge Note",
}

judge_display_df = (
    judge_df[judge_df["section"].isin(EXPECTED_SECTIONS)]
    .copy()
    .sort_values(["section", "section_score"], ascending=[True, False])
)
judge_display_df = judge_display_df[list(judge_display_columns)].rename(columns=judge_display_columns)

section_metric_directions = {
    "Section Score": "max",
    "Quality": "max",
    "Safety": "max",
    "Faithfulness": "max",
    "Numerical Accuracy": "max",
    "Trend Accuracy": "max",
    "Reasoning": "max",
    "Completeness": "max",
    "Hallucination": "min",
}

def highlight_section_best_values(column: pd.Series) -> list[str]:
    direction = section_metric_directions.get(column.name)
    if direction is None:
        return [""] * len(column)

    values = pd.to_numeric(column, errors="coerce")
    if values.notna().sum() == 0:
        return [""] * len(column)

    best_value = values.min() if direction == "min" else values.max()
    return [section_best_style if pd.notna(value) and value == best_value else "" for value in values]

judge_percent_columns = ["Section Score", "Quality", "Safety"]
judge_raw_score_columns = [
    "Faithfulness",
    "Numerical Accuracy",
    "Trend Accuracy",
    "Reasoning",
    "Completeness",
    "Hallucination",
]

judge_styler = (
    judge_display_df.style
    .format({col: "{:.1%}" for col in judge_percent_columns})
    .format({col: "{:.1f}" for col in judge_raw_score_columns})
    .apply(highlight_section_best_values, axis=0)
    .set_properties(subset=["Judge Note"], **{"max-width": "520px", "white-space": "normal"})
    .set_caption("Section-Level Judge Scores - best values highlighted in gold")
)

display(judge_styler)

section_winners_df = (
    judge_df[judge_df["section"].isin(EXPECTED_SECTIONS)]
    .sort_values(["section", "section_score"], ascending=[True, False])
    .groupby("section", as_index=False)
    .first()
)

print("Section winners")
for _, row in section_winners_df.iterrows():
    print(f"- {row['section']}: {row['model_label']} ({row['section_score']:.1%}).")

weakest_section_rows = []
for model_label, model_sections in judge_df[judge_df["section"].isin(EXPECTED_SECTIONS)].groupby("model_label"):
    weakest = model_sections.sort_values("section_score", ascending=True).head(2)
    for position, (_, row) in enumerate(weakest.iterrows(), start=1):
        weakest_section_rows.append({
            "model_label": model_label,
            "weakness_rank": position,
            "section": row["section"],
            "section_score": row["section_score"],
            "quality_score": row["quality_score"],
            "safety_score": row["safety_score"],
            "justification": row["justification"],
        })

weakest_sections_df = pd.DataFrame(weakest_section_rows)

print()
print("Weakest sections by model")
for model_label, weakest in weakest_sections_df.groupby("model_label"):
    weakness_text = "; ".join(
        f"{row['section']} ({row['section_score']:.1%})"
        for _, row in weakest.iterrows()
    )
    print(f"- {model_label}: {weakness_text}.")


,Model,Section,Section Score,Quality,Safety,Faithfulness,Numerical Accuracy,Trend Accuracy,Reasoning,Completeness,Hallucination,Judge Note
19,Phi-4,Asset Quality and Risk Profile,0.878000,0.860000,0.950000,9.0,9.5,9.0,8.0,7.5,0.5,All risk metrics match the ground truth. Reasoning is solid; completeness is slightly reduced by omission of provisioning discipline.
67,Phi-4,Asset Quality and Risk Profile,0.876000,0.870000,0.900000,9.0,9.5,9.0,8.0,8.0,1.0,"All key metrics and trends are correctly reported. Macro‑risk context is missing, slightly affecting completeness."
35,Phi-4,Asset Quality and Risk Profile,0.858000,0.860000,0.850000,9.0,9.5,9.0,7.5,8.0,1.5,"The NPL, coverage ratio, and cost of risk figures match the ground truth. The section faithfully represents the risk profile and trend, with minimal hallucination and good completeness."
51,Phi-4,Asset Quality and Risk Profile,0.818000,0.910000,0.450000,9.5,9.5,9.5,8.0,9.0,5.5,"All NPL, coverage ratio, and cost‑of‑risk figures match the ground truth. The reasoning is sound and the section is largely complete, with no hallucinations."
43,Llama 3.1,Asset Quality and Risk Profile,0.680000,0.650000,0.800000,7.0,7.5,7.0,5.5,5.5,2.0,"The NPL trend and coverage ratio changes are consistent with the ground truth. The CET1 growth rate of 8.7% aligns with the 11.5% to 12.5% increase. The section omits provisioning discipline details, slightly reducing completeness, and the growth rate phrasing is a minor hallucination."
59,Llama 3.1,Asset Quality and Risk Profile,0.650000,0.600000,0.850000,6.5,7.0,6.5,5.5,4.5,1.5,"The NPL decline and coverage ratio figures are consistent with the ground truth, though the coverage ratio is slightly lower than the ~70% mentioned. The cost‑of‑risk trend is omitted, reducing completeness. No hallucinations are present."
3,Phi-4,Asset Quality and Risk Profile,0.640000,0.700000,0.400000,7.5,8.0,7.5,6.5,5.5,6.0,"The NPL trend and cost‑of‑risk decline are correctly reported. The coverage ratio drop is a minor discrepancy but not contradictory. The section lacks mention of macro pressures and the slight NPL uptick in 2024, reducing completeness. No hallucinations."
75,Llama 3.1,Asset Quality and Risk Profile,0.586000,0.520000,0.850000,6.0,6.0,6.0,4.5,3.5,1.5,"The model captures the general stability of NPLs and the CET1 increase but misstates the exact NPL trend and coverage ratio. It omits provisioning discipline and cost of risk details, lowering completeness. No hallucinated data."
27,Llama 3.1,Asset Quality and Risk Profile,0.556000,0.520000,0.700000,6.0,7.0,5.0,4.0,4.0,3.0,The model output provides some numerical data but misses important contextual information and explanations regarding asset quality.
11,Llama 3.1,Asset Quality and Risk Profile,0.526000,0.520000,0.550000,6.5,6.0,5.0,4.5,4.0,4.5,"The model correctly cites the NPL trend but misstates the coverage ratio and cost of risk trend, introducing hallucinations. Faithfulness is moderate."


Section winners
- Asset Quality and Risk Profile: Phi-4 (87.8%).
- Balance Sheet Structure and Liquidity: Phi-4 (91.8%).
- Capital Adequacy: Phi-4 (96.8%).
- Conclusion: Phi-4 (78.0%).
- Executive Summary: Phi-4 (79.6%).
- Key Risks and Watch Points: Phi-4 (66.4%).
- Profitability and Operational Efficiency: Phi-4 (87.8%).
- Revenue Dynamics: Phi-4 (87.6%).

Weakest sections by model
- Llama 3.1: Key Risks and Watch Points (10.0%); Key Risks and Watch Points (10.0%).
- Phi-4: Key Risks and Watch Points (39.4%); Key Risks and Watch Points (50.4%).


## Aggregate judge scores by model


In [57]:
aggregated_judge_scores = (
    judge_df.groupby("model_label")[judge_metrics]
    .mean()
    .reset_index()
)

display(aggregated_judge_scores)

agreement_mean = np.nan
if use_ensemble_judge and "judge_agreement" in outputs_df.columns:
    agreement_mean = outputs_df["judge_agreement"].mean()
    print(f"\nInter-judge agreement: {agreement_mean:.3f}")
    print("(> 0.85 = reliable  |  < 0.70 = review judge prompts)")


,model_label,faithfulness,numerical_accuracy,trend_accuracy,reasoning_quality,completeness,hallucination
0,Llama 3.1,6.0125,6.3750,6.2125,4.8250,4.1750,2.6375
1,Phi-4,8.1625,8.2625,8.2875,7.2625,6.8625,3.0000



Inter-judge agreement: 0.818
(> 0.85 = reliable  |  < 0.70 = review judge prompts)


## Score model
The final score is a weighted 0-1 score designed for model comparison. Judge quality carries the most weight because factuality, numerical accuracy, trend accuracy, reasoning, and completeness matter more than lexical overlap for financial reporting. Semantic similarity captures reference alignment, structure rewards complete section coverage, safety rewards low hallucination, and efficiency rewards lower latency.


---
## 🔢 Layer 3 — Domain numerical accuracy scorer

This is the only metric that directly verifies **financial numbers**.
ROUGE and BERTScore give the same score whether a model says "CET1 13.1%" or "CET1 17.3%"
— this scorer does not.

**Method:** Regex extracts every percentage, currency amount, and multiple from both the
prediction and reference. For each reference number, it checks whether the prediction
contains a matching value within **±2% relative tolerance**.

`score = matched_numbers / total_reference_numbers` → range 0–1.

Unmatched numbers (likely hallucinated or missed) are logged as a JSON artifact to MLflow
for debugging.

In [58]:
import re

# Patterns that cover the financial report format used in BIAT analysis
_NUM_PATTERNS = [
    r"(\d{1,3}(?:[,\s]\d{3})*(?:\.\d+)?)\s*(?:TND|MAD|EGP|BHD|DZD|LYD|AED|JOD)\s*(?:millions?|bn)?",
    r"(\d{1,3}(?:\.\d+)?)\s*%",            # percentages: 13.1%
    r"(\d{1,3}(?:\.\d+)?)\s*x\b",          # multiples: 3.2x
    r"\b(\d{1,4}(?:\.\d+)?)\s*(?:million|billion|bn)\b",
]

def extract_financial_numbers(text: str) -> list[float]:
    """Return all numeric values found in a financial report text."""
    found = []
    for pattern in _NUM_PATTERNS:
        for match in re.finditer(pattern, text, re.IGNORECASE):
            raw = match.group(1).replace(",", "").replace(" ", "")
            try:
                found.append(float(raw))
            except ValueError:
                continue
    return found


def numerical_accuracy_score(prediction: str, reference: str, tolerance: float = 0.02) -> tuple[float, list[float]]:
    """
    Fraction of reference numbers that appear in prediction within +/- tolerance.
    tolerance=0.02 means +/- 2% relative difference is accepted.
    Returns (score, unmatched_numbers). Score is 1.0 if reference has no extractable numbers.
    """
    ref_nums = extract_financial_numbers(reference)
    pred_nums = extract_financial_numbers(prediction)

    if not ref_nums:
        return 1.0, []

    matched = 0
    unmatched = []
    for ref_val in ref_nums:
        denom = abs(ref_val) if abs(ref_val) > 1e-9 else 1.0
        if any(abs(p - ref_val) / denom <= tolerance for p in pred_nums):
            matched += 1
        else:
            unmatched.append(ref_val)

    score = round(matched / len(ref_nums), 4)
    return score, unmatched   # return tuple so caller can log unmatched list


# Apply to outputs_df
numerical_results = outputs_df.apply(
    lambda row: numerical_accuracy_score(
        row["output_text_normalized"], row["reference_output_normalized"]
    ),
    axis=1,
)
outputs_df["numerical_accuracy_det"] = [r[0] for r in numerical_results]
outputs_df["numerical_unmatched"]    = [r[1] for r in numerical_results]

# Display summary
display(
    outputs_df.groupby("model_label")[["numerical_accuracy_det"]]
    .agg(["mean", "std", "min", "max"])
    .round(4)
)

# Log per-model
for model_label, group in outputs_df.groupby("model_label"):
    mlflow.log_metric(
        f"numerical_accuracy_det_{model_label}_mean",
        group["numerical_accuracy_det"].mean()
    )
    mlflow.log_metric(
        f"numerical_accuracy_det_{model_label}_std",
        group["numerical_accuracy_det"].std()
    )
    # Log the unmatched numbers list as an artifact for debugging
    unmatched_flat = [v for lst in group["numerical_unmatched"] for v in lst]
    mlflow.log_dict(
        {"unmatched_numbers": unmatched_flat, "model": model_label},
        f"numerical_unmatched_{model_label}.json"
    )


numerical_accuracy_det                        
                              mean     std     min     max
model_label                                               
Llama 3.1                   0.6421  0.2151  0.4211  0.8947
Phi-4                       0.8105  0.0600  0.7368  0.8947

## RAGAS evaluation


---
## 🧬 Layer 2 — RAGAS evaluation

RAGAS provides three complementary metrics that go beyond lexical overlap:

| Metric | Method | What it catches |
|---|---|---|
| **SemanticSimilarity** | Local HuggingFace embeddings (no API cost) | Overall meaning alignment |
| **AnswerCorrectness** | 30% LLM factual + 70% embedding similarity, section-by-section | Per-section content accuracy |
| **FactualCorrectness** | NLI claim decomposition via LLM, section-by-section | Atomic fact verification |

**Token cost note:** FactualCorrectness and AnswerCorrectness each make multiple LLM calls
per section. To stay within Groq's daily token limit, RAGAS uses `GROQ_JUDGE_MODEL_2`
(separate quota from the main judge) and evaluates only 1 run per model by default
(`RAGAS_RUNS_PER_MODEL = 1`).

Running section-by-section (rather than on the full report) gives diagnostic precision:
you can see which specific sections a model handles well vs poorly.

In [59]:
from ragas.run_config import RunConfig
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_huggingface import HuggingFaceEmbeddings

outputs_ragas = outputs_df.reset_index(drop=True).copy()
outputs_ragas["sample_id"] = outputs_ragas.index

# RAGAS AnswerCorrectness and FactualCorrectness are token-heavy.
# Default: evaluate 1 generated report per model, not all 5 runs.
# Override in .env if needed:
# - RAGAS_RUNS_PER_MODEL=2 evaluates two runs per model.
# - RAGAS_TOTAL_RUNS=1 evaluates one report total across all models.
# - RAGAS_TOTAL_RUNS=0 keeps the per-model sampling behavior.
# RAGAS always uses GROQ_JUDGE_MODEL_2 from .env.

RAGAS_TOTAL_RUNS = 0
RAGAS_RUNS_PER_MODEL = 1
RAGAS_JUDGE_MODEL = os.getenv("GROQ_JUDGE_MODEL_2")
if not RAGAS_JUDGE_MODEL:
    raise ValueError("GROQ_JUDGE_MODEL_2 must be set in .env for RAGAS evaluation.")



ragas_eval_df = (
    outputs_ragas
    .sort_values(["model_label", "sample_id"])
    .groupby("model_label", group_keys=False)
    .head(RAGAS_RUNS_PER_MODEL)
    .reset_index(drop=True)
    if RAGAS_RUNS_PER_MODEL > 0
    else outputs_ragas.copy()
)

if RAGAS_TOTAL_RUNS > 0:
    ragas_eval_df = (
        ragas_eval_df
        .sort_values(["sample_id", "model_label"])
        .head(RAGAS_TOTAL_RUNS)
        .reset_index(drop=True)
    )

print(
    f"RAGAS section metrics will score {len(ragas_eval_df)} generated report(s). "
    f"RAGAS_RUNS_PER_MODEL={RAGAS_RUNS_PER_MODEL}, "
    f"RAGAS_TOTAL_RUNS={RAGAS_TOTAL_RUNS}, "
    f"RAGAS_JUDGE_MODEL={RAGAS_JUDGE_MODEL}, "
)

if "financial_data" not in globals():
    if "financial_data" not in outputs_ragas.columns or outputs_ragas["financial_data"].dropna().empty:
        raise ValueError("outputs_df is missing financial_data. Re-run Notebook 02.")
    financial_data = outputs_ragas["financial_data"].dropna().iloc[0]

_groq_lc = ChatOpenAI(
    base_url=JUDGE_BASE_URL,
    api_key=JUDGE_API_KEY,
    model=RAGAS_JUDGE_MODEL,
    temperature=0.0,
    max_completion_tokens=1536,
    timeout=600,
    max_retries=2,
)

ragas_llm = LangchainLLMWrapper(_groq_lc)

_hf_embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

ragas_embeddings = LangchainEmbeddingsWrapper(_hf_embeddings)

run_config = RunConfig(
    timeout=600,
    max_workers=1,
)

RAGAS section metrics will score 2 generated report(s). RAGAS_RUNS_PER_MODEL=1, RAGAS_TOTAL_RUNS=0, RAGAS_JUDGE_MODEL=meta-llama/llama-4-scout-17b-16e-instruct, 


C:\Users\BRHN\AppData\Local\Temp\ipykernel_300\2833309327.py:64: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(_groq_lc)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
C:\Users\BRHN\AppData\Local\Temp\ipykernel_300\2833309327.py:70: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(_hf_embeddings)


## SemanticSimilarity only

In [60]:
semantic_rows = []

for model_label, group in ragas_eval_df.groupby("model_label"):
    samples = [
        SingleTurnSample(
            user_input=financial_data,
            response=row["output_text"],
            reference=row["reference_output"],
        )
        for _, row in group.iterrows()
    ]

    dataset_ragas = EvaluationDataset(samples=samples)

    results = ragas_evaluate(
        dataset=dataset_ragas,
        metrics=[
            SemanticSimilarity(embeddings=ragas_embeddings)
        ],
        embeddings=ragas_embeddings,
        run_config=run_config,
        raise_exceptions=False,
    )

    df = results.to_pandas()
    df["model_label"] = model_label
    df["sample_id"] = group["sample_id"].values

    semantic_rows.append(df)

if semantic_rows:
    semantic_df = pd.concat(semantic_rows, ignore_index=True)
    semantic_df.columns = [c.lower().strip().replace(" ", "_") for c in semantic_df.columns]
else:
    semantic_df = pd.DataFrame(columns=["model_label", "sample_id", "semantic_similarity"])

display(semantic_df.head())

Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]

,user_input,response,reference,semantic_similarity,model_label,sample_id
0,\n# Financial Analysis – Banque Internationale...,Executive Summary\n\nBanque Internationale Ara...,"Executive Summary\nOver the 2021–2024 period, ...",0.786305,Llama 3.1,1
1,\n# Financial Analysis – Banque Internationale...,**Executive Summary**\n\nBanque Internationale...,"Executive Summary\nOver the 2021–2024 period, ...",0.830056,Phi-4,0


## AnswerCorrectness section by section

In [61]:
CANONICAL_SECTIONS = [
    "Executive Summary",
    "Profitability and Operational Efficiency",
    "Revenue Dynamics",
    "Asset Quality and Risk Profile",
    "Balance Sheet Structure and Liquidity",
    "Capital Adequacy",
    "Key Risks and Watch Points",
    "Conclusion",
]

SECTION_ORDER = {section: idx for idx, section in enumerate(CANONICAL_SECTIONS)}


def _load_sections_json(value):
    if isinstance(value, dict):
        return value
    if pd.isna(value):
        return {}
    try:
        parsed = json.loads(value)
    except (TypeError, json.JSONDecodeError):
        return {}
    return parsed if isinstance(parsed, dict) else {}


def _canonicalize_sections(sections):
    return {
        section: str(sections.get(section, "") or "").strip()
        for section in CANONICAL_SECTIONS
    }


def extract_sections_simple(text, sections=CANONICAL_SECTIONS):
    text = str(text or "")
    extracted = {}

    heading_patterns = [
        (
            section,
            re.compile(
                rf"(?im)^\s*(?:#{{1,6}}\s*)?(?:\*\*)?\s*"
                rf"(?:\d+(?:\.\d+)*[.)]?\s*)?{re.escape(section)}"
                rf"\s*:?\s*(?:\*\*)?\s*$"
            ),
        )
        for section in sections
    ]

    matches = []
    for section, pattern in heading_patterns:
        match = pattern.search(text)
        if match:
            matches.append((match.start(), match.end(), section))

    matches.sort(key=lambda item: item[0])
    match_by_section = {section: (start, end) for start, end, section in matches}

    for section in sections:
        if section not in match_by_section:
            extracted[section] = ""
            continue

        _, start_content = match_by_section[section]
        following_starts = [start for start, _, _ in matches if start > start_content]
        end_content = min(following_starts) if following_starts else len(text)
        extracted[section] = text[start_content:end_content].strip()

    return extracted


def get_sections_from_row(row, text_col, json_col):
    sections = _load_sections_json(row.get(json_col))
    if sections:
        return _canonicalize_sections(sections)
    return _canonicalize_sections(extract_sections_simple(row.get(text_col, "")))

In [ ]:
answer_section_rows = []

for _, row in ragas_eval_df.iterrows():
    response_sections = get_sections_from_row(row, "output_text", "parsed_sections_json")
    reference_sections = get_sections_from_row(row, "reference_output", "reference_sections_json")

    samples = []
    section_names = []

    for section in CANONICAL_SECTIONS:
        response_section = response_sections.get(section, "").strip()
        reference_section = reference_sections.get(section, "").strip()

        if response_section and reference_section:
            samples.append(
                SingleTurnSample(
                    user_input=f"Compare the generated '{section}' section with the reference '{section}' section.",
                    response=response_section,
                    reference=reference_section,
                )
            )
            section_names.append(section)

    if not samples:
        continue

    dataset_ragas = EvaluationDataset(samples=samples)

    results = ragas_evaluate(
        dataset=dataset_ragas,
        metrics=[
            AnswerCorrectness(
                llm=ragas_llm,
                embeddings=ragas_embeddings,
                weights=[0.3, 0.7],  # 30% factual LLM, 70% semantic embedding
            )
        ],
        llm=ragas_llm,
        embeddings=ragas_embeddings,
        run_config=run_config,
        raise_exceptions=False,
    )

    df = results.to_pandas()
    df["model_label"] = row["model_label"]
    df["sample_id"] = row["sample_id"]
    df["section"] = section_names

    answer_section_rows.append(df)

if answer_section_rows:
    answer_section_df = pd.concat(answer_section_rows, ignore_index=True)
    answer_section_df.columns = [
        re.sub(r"\(.*?\)", "", c.lower().strip().replace(" ", "_"))
        for c in answer_section_df.columns
    ]
else:
    answer_section_df = pd.DataFrame(
        columns=["model_label", "sample_id", "section", "answer_correctness"]
    )

answer_section_df["section_order"] = answer_section_df["section"].map(SECTION_ORDER)
answer_section_df = answer_section_df.sort_values(
    ["model_label", "sample_id", "section_order"]
).reset_index(drop=True)

display(answer_section_df)

Evaluating:   0%|          | 0/8 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/8 [00:00<?, ?it/s]

In [ ]:
answer_report_df = (
    answer_section_df
    .groupby(["model_label", "sample_id"], as_index=False)
    .agg(
        answer_correctness=("answer_correctness", "mean"),
        answer_sections_scored=("section", "nunique"),
    )
)

answer_by_section_df = (
    answer_section_df
    .groupby(["model_label", "section", "section_order"])["answer_correctness"]
    .agg(["mean", "std", "count"])
    .reset_index()
    .sort_values(["model_label", "section_order"])
)

display(answer_report_df)
display(answer_by_section_df)

,model_label,sample_id,answer_correctness,answer_sections_scored
0,Llama 3.1,1,0.635191,8
1,Phi-4,0,0.644741,8


,model_label,section,section_order,mean,std,count
4,Llama 3.1,Executive Summary,0,0.546119,NaN,1
6,Llama 3.1,Profitability and Operational Efficiency,1,0.617125,NaN,1
7,Llama 3.1,Revenue Dynamics,2,0.779247,NaN,1
0,Llama 3.1,Asset Quality and Risk Profile,3,0.762853,NaN,1
1,Llama 3.1,Balance Sheet Structure and Liquidity,4,0.603881,NaN,1
2,Llama 3.1,Capital Adequacy,5,0.654152,NaN,1
5,Llama 3.1,Key Risks and Watch Points,6,0.502291,NaN,1
3,Llama 3.1,Conclusion,7,0.615864,NaN,1
12,Phi-4,Executive Summary,0,NaN,NaN,0
14,Phi-4,Profitability and Operational Efficiency,1,0.575369,NaN,1


## FactualCorrectness section by section

In [ ]:
factual_section_rows = []

for _, row in ragas_eval_df.iterrows():
    response_sections = get_sections_from_row(row, "output_text", "parsed_sections_json")
    reference_sections = get_sections_from_row(row, "reference_output", "reference_sections_json")

    samples = []
    section_names = []

    for section in CANONICAL_SECTIONS:
        response_section = response_sections.get(section, "").strip()
        reference_section = reference_sections.get(section, "").strip()

        if response_section and reference_section:
            samples.append(
                SingleTurnSample(
                    user_input=f"Assess factual correctness for the generated '{section}' section against the reference '{section}' section.",
                    response=response_section,
                    reference=reference_section,
                )
            )
            section_names.append(section)

    if not samples:
        continue

    dataset_ragas = EvaluationDataset(samples=samples)

    results = ragas_evaluate(
        dataset=dataset_ragas,
        metrics=[
            FactualCorrectness(
                llm=ragas_llm,
                atomicity="low",
                coverage="low",
            )
        ],
        llm=ragas_llm,
        run_config=run_config,
        raise_exceptions=False,
    )

    df = results.to_pandas()
    df["model_label"] = row["model_label"]
    df["sample_id"] = row["sample_id"]
    df["section"] = section_names

    factual_section_rows.append(df)

if factual_section_rows:
    factual_section_df = pd.concat(factual_section_rows, ignore_index=True)
    factual_section_df.columns = [
        re.sub(r"\(.*?\)", "", c.lower().strip().replace(" ", "_"))
        for c in factual_section_df.columns
    ]
else:
    factual_section_df = pd.DataFrame(
        columns=["model_label", "sample_id", "section", "factual_correctness"]
    )

factual_section_df["section_order"] = factual_section_df["section"].map(SECTION_ORDER)
factual_section_df = factual_section_df.sort_values(
    ["model_label", "sample_id", "section_order"]
).reset_index(drop=True)

display(factual_section_df)

Evaluating:   0%|          | 0/8 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/8 [00:00<?, ?it/s]

,user_input,response,reference,factual_correctness,model_label,sample_id,section,section_order
0,Assess factual correctness for the generated '...,Banque Internationale Arabe de Tunisie (BIAT) ...,"Over the 2021-2024 period, BIAT demonstrates a...",0.18,Llama 3.1,1,Executive Summary,0
1,Assess factual correctness for the generated '...,The bank's profitability has improved signific...,Profitability has strengthened steadily across...,0.40,Llama 3.1,1,Profitability and Operational Efficiency,1
2,Assess factual correctness for the generated '...,Net Banking Income (NBI) has grown by 38% from...,Net Banking Income (NBI) grew consistently ove...,0.15,Llama 3.1,1,Revenue Dynamics,2
3,Assess factual correctness for the generated '...,The bank's Non-Performing Loans (NPL) ratio ha...,The bank’s risk profile shows overall improvem...,0.50,Llama 3.1,1,Asset Quality and Risk Profile,3
4,Assess factual correctness for the generated '...,The bank's total assets have grown by 25% from...,BIAT maintains a sound and conservative balanc...,0.31,Llama 3.1,1,Balance Sheet Structure and Liquidity,4
5,Assess factual correctness for the generated '...,The bank's Common Equity Tier 1 (CET1) ratio h...,"Capitalization has improved gradually, with th...",0.62,Llama 3.1,1,Capital Adequacy,5
6,Assess factual correctness for the generated '...,"The bank's loan growth has been significant, w...","Despite the overall strong performance, severa...",0.29,Llama 3.1,1,Key Risks and Watch Points,6
7,Assess factual correctness for the generated '...,Banque Internationale Arabe de Tunisie (BIAT) ...,BIAT’s financial profile over 2021-2024 reflec...,0.42,Llama 3.1,1,Conclusion,7
8,Assess factual correctness for the generated '...,Banque Internationale Arabe de Tunisie (BIAT) ...,"Over the 2021-2024 period, BIAT demonstrates a...",0.47,Phi-4,0,Executive Summary,0
9,Assess factual correctness for the generated '...,"BIAT's profitability metrics, including Return...",Profitability has strengthened steadily across...,0.25,Phi-4,0,Profitability and Operational Efficiency,1


## Combine results

In [ ]:
def clean_ragas_metric_col(col: str) -> str:
    col = col.lower().strip().replace(" ", "_")
    return re.sub(r"\(.*?\)", "", col)

factual_section_df = factual_section_df.rename(columns=clean_ragas_metric_col)
answer_section_df = answer_section_df.rename(columns=clean_ragas_metric_col)

factual_report_df = (
    factual_section_df
    .groupby(["model_label", "sample_id"], as_index=False)
    .agg(
        factual_correctness=("factual_correctness", "mean"),
        factual_sections_scored=("section", "nunique"),
    )
)

factual_by_section_df = (
    factual_section_df
    .groupby(["model_label", "section", "section_order"])["factual_correctness"]
    .agg(["mean", "std", "count"])
    .reset_index()
    .sort_values(["model_label", "section_order"])
)

ragas_section_combined_df = (
    answer_section_df[["model_label", "sample_id", "section", "section_order", "answer_correctness"]]
    .merge(
        factual_section_df[["model_label", "sample_id", "section", "factual_correctness"]],
        on=["model_label", "sample_id", "section"],
        how="outer",
    )
    .sort_values(["model_label", "sample_id", "section_order"])
    .reset_index(drop=True)
)

# display(factual_report_df)
# display(factual_by_section_df)
# display(ragas_section_combined_df)

In [ ]:
semantic_keep = semantic_df[["model_label", "sample_id", "semantic_similarity"]]
base_keep = ragas_eval_df[["model_label", "sample_id"]].drop_duplicates()

ragas_combined_df = (
    base_keep
    .merge(semantic_keep, on=["model_label", "sample_id"], how="left")
    .merge(answer_report_df, on=["model_label", "sample_id"], how="left")
    .merge(factual_report_df, on=["model_label", "sample_id"], how="left")
)

ragas_combined_df["ragas_mean_score"] = ragas_combined_df[
    ["semantic_similarity", "answer_correctness", "factual_correctness"]
].mean(axis=1)

ragas_combined_df = ragas_combined_df.sort_values(
    ["model_label", "sample_id"]
).reset_index(drop=True)

ragas_df = ragas_combined_df.copy()

display(ragas_combined_df)

,model_label,sample_id,semantic_similarity,answer_correctness,answer_sections_scored,factual_correctness,factual_sections_scored,ragas_mean_score
0,Llama 3.1,1,0.786305,0.635191,8,0.35875,8,0.593415
1,Phi-4,0,0.830056,0.644741,8,0.38875,8,0.621182


In [ ]:
ragas_metric_cols = [
    col
    for col in ["semantic_similarity", "answer_correctness", "factual_correctness"]
    if col in ragas_combined_df.columns
]

def std_or_zero(values: pd.Series) -> float:
    values = pd.to_numeric(values, errors="coerce").dropna()
    return 0.0 if len(values) <= 1 else values.std()

ragas_summary_parts = []

if "semantic_similarity" in ragas_combined_df.columns and not ragas_combined_df.empty:
    semantic_summary = ragas_combined_df.groupby("model_label").agg(
        semantic_similarity_mean=("semantic_similarity", "mean"),
        semantic_similarity_std=("semantic_similarity", std_or_zero),
    )
    ragas_summary_parts.append(semantic_summary)

section_ragas_metric_cols = [
    col
    for col in ["answer_correctness", "factual_correctness"]
    if col in ragas_section_combined_df.columns
]

if section_ragas_metric_cols and not ragas_section_combined_df.empty:
    section_summary = ragas_section_combined_df.groupby("model_label").agg(
        **{
            f"{metric}_mean": (metric, "mean")
            for metric in section_ragas_metric_cols
        },
        **{
            f"{metric}_std": (metric, std_or_zero)
            for metric in section_ragas_metric_cols
        },
    )
    ragas_summary_parts.append(section_summary)

ragas_summary = (
    pd.concat(ragas_summary_parts, axis=1).round(4)
    if ragas_summary_parts
    else pd.DataFrame()
)

display(ragas_summary)

,semantic_similarity_mean,semantic_similarity_std,answer_correctness_mean,factual_correctness_mean,answer_correctness_std,factual_correctness_std
model_label,,,,,,
Llama 3.1,0.7863,0.0,0.6352,0.3588,0.0961,0.1587
Phi-4,0.8301,0.0,0.6447,0.3888,0.0921,0.2406


---
## 📊 Variance summary — all metric layers

Aggregates all metrics across the 5 temperature runs into **mean ± std** per model.
Gold highlight = best mean value across models for that metric.

**How to read this table:**
- A model with `mean=0.84, std=0.01` is consistent and reliable at that metric
- A model with `mean=0.84, std=0.09` is unreliable — the average looks good but
  individual outputs vary widely; users would occasionally get a poor-quality result
- The `std` column is directly informative for deployment decisions even on 1 test case

**Important caveat:** These scores are from a single financial report (BIAT 2021–2024).
The mean/std reflects sensitivity to temperature, not generalization across bank profiles.
Scale to ≥20 diverse test cases before drawing deployment conclusions.

In [ ]:
if "std_or_zero" not in globals():
    def std_or_zero(values: pd.Series) -> float:
        values = pd.to_numeric(values, errors="coerce").dropna()
        return 0.0 if len(values) <= 1 else values.std()

METRIC_LAYERS = {
    "Layer 1 — Deterministic": ["rougeL", "chrf", "bert_f1", "length_score"],
    "Layer 2 — RAGAS":         ["factual_correctness", "answer_correctness", "semantic_similarity"],
    "Layer 3 — Domain":        ["numerical_accuracy_det"],
    "Layer 4 — LLM judge":     ["judge_quality_score", "safety_score", "faithfulness",
                                 "numerical_accuracy", "trend_accuracy", "reasoning_quality",
                                 "completeness"],
    "Operational":             ["latency_sec"],
}

variance_rows = []

for layer_name, metrics in METRIC_LAYERS.items():
    for metric in metrics:
        if metric not in outputs_df.columns:
            continue
        for model_label, group in outputs_df.groupby("model_label"):
            vals = pd.to_numeric(group[metric], errors="coerce").dropna()
            if len(vals) == 0:
                continue
            variance_rows.append({
                "layer":        layer_name,
                "metric":       metric,
                "model":        model_label,
                "mean":         round(vals.mean(), 4),
                "std":          round(std_or_zero(vals), 4),
                "min":          round(vals.min(), 4),
                "max":          round(vals.max(), 4),
                "n_runs":       len(vals),
            })

# Add RAGAS rows from ragas_df if it exists
if not ragas_df.empty:
    for metric in ["factual_correctness", "answer_correctness", "semantic_similarity"]:
        if metric not in ragas_df.columns:
            continue
        for model_label, group in ragas_df.groupby("model_label"):
            vals = pd.to_numeric(group[metric], errors="coerce").dropna()
            variance_rows.append({
                "layer": "Layer 2 — RAGAS", "metric": metric, "model": model_label,
                "mean": round(vals.mean(), 4), "std": round(std_or_zero(vals), 4),
                "min": round(vals.min(), 4), "max": round(vals.max(), 4),
                "n_runs": len(vals),
            })

variance_df = pd.DataFrame(variance_rows)

# Pivot: models as columns, metric+layer as rows
pivot = variance_df.pivot_table(
    index=["layer", "metric"],
    columns="model",
    values=["mean", "std"],
    aggfunc="first",
).round(4)

variance_good_style = "background-color: #fff2cc; color: #7a4f00; font-weight: 700;"
LOWER_MEAN_IS_BETTER = {"latency_sec"}

def highlight_variance_best_mean(row: pd.Series) -> list[str]:
    styles = [""] * len(row)
    metric_name = row.name[1] if isinstance(row.name, tuple) else row.name

    mean_cols = [col for col in row.index if col[0] == "mean"]
    values = pd.to_numeric(row[mean_cols], errors="coerce").dropna()
    if values.empty:
        return styles

    if metric_name in LOWER_MEAN_IS_BETTER:
        best_value = values.min()
    else:
        best_value = values.max()

    for col in mean_cols:
        if pd.notna(row[col]) and row[col] == best_value:
            styles[row.index.get_loc(col)] = variance_good_style

    return styles

variance_styler = (
    pivot.style
    .format(precision=4, na_rep="0.0000")
    .apply(highlight_variance_best_mean, axis=1)
    .set_caption(
        f"Metric variance summary — {N_RUNS} runs per model (single case, directional only); gold = best mean across models"
    )
)

display(variance_styler)

# Log to MLflow
mlflow.log_table(variance_df, artifact_file="variance_summary.json")


---
## ⏱️ Operational metrics — latency, tokens, cost

Production readiness metrics. These are independent of output quality and reflect the
real-world cost of deploying each model.

| Metric | Why it matters |
|---|---|
| **Latency (sec)** | User-facing response time — Groq is significantly faster than Azure |
| **Prompt tokens** | Fixed per model (same input) — reflects context window efficiency |
| **Completion tokens** | Variable — longer outputs cost more and may indicate verbosity |
| **Estimated cost (USD)** | Per-analysis cost estimate based on provider pricing |

Cost estimates use approximate per-token prices. Update `TOKEN_PRICE_PER_1K` in the cell
to match your actual Azure and Groq plan rates.

In [ ]:
OPERATIONAL_METRICS = ["latency_sec", "prompt_tokens", "completion_tokens", "total_tokens"]

ops_rows = []
for model_label, group in outputs_df.groupby("model_label"):
    for metric in OPERATIONAL_METRICS:
        if metric not in group.columns:
            continue
        vals = pd.to_numeric(group[metric], errors="coerce").dropna()
        if len(vals) == 0:
            continue
        ops_rows.append({
            "model": model_label,
            "metric": metric,
            "mean": round(vals.mean(), 3),
            "std": round(vals.std(), 3),
            "min": round(vals.min(), 3),
            "max": round(vals.max(), 3),
        })
        mlflow.log_metric(f"ops_{metric}_{model_label}_mean", vals.mean())
        mlflow.log_metric(f"ops_{metric}_{model_label}_std", vals.std())

ops_df = pd.DataFrame(ops_rows)

# Cost estimate (approximate — adjust price per token to match your plan)
TOKEN_PRICE_PER_1K = {
    "phi4":  0.00014,   # adjust to your Azure pricing
    "llama": 0.00006,   # adjust to your Groq pricing
}
for _, row in ops_df[ops_df["metric"] == "total_tokens"].iterrows():
    model_key = "phi4" if "phi4" in row["model"].lower() else "llama"
    cost_per_run = row["mean"] / 1000 * TOKEN_PRICE_PER_1K.get(model_key, 0.0001)
    print(f"{row['model']}: ~${cost_per_run:.5f} per analysis run (est.)")
    mlflow.log_metric(f"ops_estimated_cost_usd_{row['model']}", cost_per_run)

display(ops_df.pivot_table(
    index="metric", columns="model", values=["mean", "std"]
).round(3).style.set_caption("Operational metrics — mean ± std across runs"))


Llama 3.1: ~$0.00010 per analysis run (est.)
Phi-4: ~$0.00011 per analysis run (est.)


---
## 🏆 Final comparison — model leaderboard

Aggregates all metric layers into a single weighted final score for head-to-head comparison.

**Weighting rationale:**
| Component | Weight | Rationale |
|---|---|---|
| Judge quality | 38% | Faithfulness, numerical accuracy, reasoning — the core task requirements |
| Semantic similarity | 19% | Reference alignment — how close the analysis is to the ideal |
| Structure | 14.25% | Section completeness — the report format is a hard requirement |
| Safety (non-hallucination) | 14.25% | Hallucination is high-risk in financial reporting |
| Efficiency | 9.5% | Latency score — relevant for production deployment |
| ChrF++ | 5% | Character-level lexical quality — independent signal from semantic layer |

Gold cells = best value per column across models.

In [ ]:
comparison_agg_cols = [
    "latency_ms",
    "rougeL",
    "bleu",
    "chrf",
    "bert_f1",
    "length_score",
    "structure_score",
    "numerical_accuracy_det",
]

comparison_df = (
    outputs_df.groupby(["model_label", "provider"], as_index=False)[comparison_agg_cols]
    .mean()
)

comparison_df = comparison_df.merge(aggregated_judge_scores, on="model_label")
comparison_df["latency_score"] = invert_minmax(comparison_df["latency_ms"])

# Keep raw judge averages in 0-10 form, and add normalized 0-1 score columns for charts/scoring.
for metric in judge_metrics:
    comparison_df[f"{metric}_score"] = (
        pd.to_numeric(comparison_df[metric], errors="coerce") / 10
    ).fillna(0.0).clip(0, 1)

# Higher hallucination is worse, so invert it for any "higher is better" visualization.
comparison_df["non_hallucination_score"] = 1 - comparison_df["hallucination_score"]

semantic_metrics = ["rougeL", "bleu", "bert_f1"]
judge_quality_metrics = [
    "faithfulness_score",
    "numerical_accuracy_score",
    "trend_accuracy_score",
    "reasoning_quality_score",
    "completeness_score",
]

comparison_df["semantic_score"] = comparison_df[semantic_metrics].mean(axis=1)
comparison_df["judge_quality_score"] = comparison_df[judge_quality_metrics].mean(axis=1)
comparison_df["safety_score"] = comparison_df["non_hallucination_score"]
comparison_df["efficiency_score"] = comparison_df["latency_score"]

comparison_df["final_score"] = sum(
    comparison_df[metric] * weight
    for metric, weight in SCORE_WEIGHTS.items()
)
comparison_df["rank"] = comparison_df["final_score"].rank(ascending=False, method="dense").astype(int)

comparison_display_columns = {
    "rank": "Rank",
    "model_label": "Model",
    "provider": "Provider",
    "final_score": "Final Score",
    "judge_quality_score": "Judge Quality",
    "semantic_score": "Semantic Similarity",
    "structure_score": "Structure",
    "safety_score": "Safety",
    "efficiency_score": "Efficiency",
    "latency_ms": "Latency (ms)",
    "rougeL": "Rouge-L",
    "bleu": "BLEU",
    "chrf": "ChrF++",
    "bert_f1": "BERTScore F1",
}

comparison_display_df = (
    comparison_df[list(comparison_display_columns)]
    .rename(columns=comparison_display_columns)
    .sort_values(["Rank", "Model"])
    .reset_index(drop=True)
)

metric_directions = {
    "Rank": "min",
    "Latency (ms)": "min",
    "Final Score": "max",
    "Judge Quality": "max",
    "Semantic Similarity": "max",
    "Structure": "max",
    "Safety": "max",
    "Efficiency": "max",
    "Rouge-L": "max",
    "BLEU": "max",
    "ChrF++": "max",
    "BERTScore F1": "max",
}

best_style = "background-color: #fff2cc; color: #7a4f00; font-weight: 700;"

def highlight_best_values(column: pd.Series) -> list[str]:
    direction = metric_directions.get(column.name)
    if direction is None:
        return [""] * len(column)

    values = pd.to_numeric(column, errors="coerce")
    if values.notna().sum() == 0:
        return [""] * len(column)

    best_value = values.min() if direction == "min" else values.max()
    return [best_style if pd.notna(value) and value == best_value else "" for value in values]

percent_columns = [
    "Final Score",
    "Judge Quality",
    "Semantic Similarity",
    "Structure",
    "Safety",
    "Efficiency",
    "Rouge-L",
    "BLEU",
    "ChrF++",
    "BERTScore F1",
]

comparison_styler = (
    comparison_display_df.style
    .format({col: "{:.1%}" for col in percent_columns})
    .format({"Latency (ms)": "{:.0f}", "Rank": "{:.0f}"})
    .apply(highlight_best_values, axis=0)
    .set_caption("Model Comparison Summary - best values highlighted in gold")
)

display(comparison_styler)

winner = comparison_df.sort_values("final_score", ascending=False).iloc[0]
fastest = comparison_df.sort_values("latency_ms", ascending=True).iloc[0]
most_accurate = comparison_df.sort_values("numerical_accuracy_score", ascending=False).iloc[0]

print("Evaluation highlights")
print(f"- Best overall: {winner['model_label']} with final score {winner['final_score']:.1%}.")
print(f"- Fastest model: {fastest['model_label']} at {fastest['latency_ms']:.0f} ms.")
print(f"- Best numerical accuracy: {most_accurate['model_label']} at {most_accurate['numerical_accuracy_score']:.1%}.")

score_columns = [
    "judge_quality_score",
    "semantic_score",
    "structure_score",
    "safety_score",
    "efficiency_score",
]
score_labels = {
    "judge_quality_score": "judge quality",
    "semantic_score": "semantic similarity",
    "structure_score": "structure",
    "safety_score": "safety",
    "efficiency_score": "efficiency",
}

for _, row in comparison_df.sort_values("rank").iterrows():
    weakest_metric = min(score_columns, key=lambda metric: row[metric])
    print(
        f"- Main weakness for {row['model_label']}: "
        f"{score_labels[weakest_metric]} ({row[weakest_metric]:.1%})."
    )


,Rank,Model,Provider,Final Score,Judge Quality,Semantic Similarity,Structure,Safety,Efficiency,Latency (ms),Rouge-L,BLEU,ChrF++,BERTScore F1
0,1,Llama 3.1,groq,0.648421,0.552000,0.413185,1.000000,0.736250,1.000000,3152,0.275027,0.084996,0.354800,0.879531
1,2,Phi-4,azure_openai_compatible,0.643493,0.776750,0.435404,1.000000,0.700000,0.000000,23793,0.295571,0.120555,0.467020,0.890087


Evaluation highlights
- Best overall: Llama 3.1 with final score 64.8%.
- Fastest model: Llama 3.1 at 3152 ms.
- Best numerical accuracy: Phi-4 at 82.6%.
- Main weakness for Llama 3.1: semantic similarity (41.3%).
- Main weakness for Phi-4: efficiency (0.0%).


---
## 📡 Visualizations

Two charts summarise the comparison:

1. **Radar chart** — shows each model's profile across all metric dimensions simultaneously.
   A model that is strong on judge quality but weak on latency will show a characteristic
   shape. Both charts are saved as HTML (interactive) and PNG (for reports).

2. **Grouped bar chart with error bars** — shows mean ± std for key metrics side by side.
   Error bars represent variation across the 5 temperature runs.
   Taller error bars = more sensitive to temperature = less reliable in production.

In [ ]:
VARIANCE_METRIC_ALIASES = {
    "faithfulness_score": ("faithfulness", lambda value: value / 10),
    "numerical_accuracy_score": ("numerical_accuracy", lambda value: value / 10),
    "trend_accuracy_score": ("trend_accuracy", lambda value: value / 10),
    "reasoning_quality_score": ("reasoning_quality", lambda value: value / 10),
    "completeness_score": ("completeness", lambda value: value / 10),
    "non_hallucination_score": ("hallucination", lambda value: 1 - (value / 10)),
}


def plot_plotly_radar(df: pd.DataFrame, model_col: str, metrics: list[str], title: str):
    categories = metrics + [metrics[0]]

    fig = go.Figure()

    for _, row in df.iterrows():
        model_label = row[model_col]
        values = []
        for metric in metrics:
            variance_metric, transform = VARIANCE_METRIC_ALIASES.get(
                metric,
                (metric, lambda value: value),
            )
            metric_mean = variance_df[
                (variance_df.model == model_label) & (variance_df.metric == variance_metric)
            ]["mean"].values
            value = transform(metric_mean[0]) if len(metric_mean) else row[metric]
            values.append(float(np.clip(value, 0, 1)))
        values += values[:1]

        fig.add_trace(
            go.Scatterpolar(
                r=values,
                theta=categories,
                fill="toself",
                name=row[model_col],
            )
        )

    fig.update_layout(
        title=title,
        polar=dict(
            radialaxis=dict(visible=True, range=[0, 1]),
        ),
        showlegend=True,
    )
    return fig


judge_radar_metrics = [
    "faithfulness_score",
    "numerical_accuracy_score",
    "trend_accuracy_score",
    "reasoning_quality_score",
    "completeness_score",
    "non_hallucination_score",
]

radar_metrics = [
    "rougeL",
    "bleu",
    "chrf",
    "bert_f1",
    "length_score",
    "structure_score",
    "latency_score",
] + judge_radar_metrics

for col in radar_metrics:
    comparison_df[col] = pd.to_numeric(comparison_df[col], errors="coerce").fillna(0.0).clip(0, 1)

radar_fig = plot_plotly_radar(
    comparison_df,
    model_col="model_label",
    metrics=radar_metrics,
    title="Evaluation Radar Chart: Llama 3.1 vs Phi-4",
)

radar_fig.show()

radar_html_path = FIGURES_DIR / f"radar_comparison_{CASE_ID}.html"
radar_png_path = FIGURES_DIR / f"radar_comparison_{CASE_ID}.png"

radar_fig.write_html(str(radar_html_path))
radar_fig.write_image(str(radar_png_path))

print("Saved:", radar_html_path)
print("Saved:", radar_png_path)

bar_metrics = ["rougeL", "chrf", "bert_f1", "numerical_accuracy_det"]
if not ragas_df.empty:
    bar_metrics += ["factual_correctness", "answer_correctness"]

fig_bar = go.Figure()
for model_label in variance_df["model"].unique():
    sub = variance_df[
        (variance_df["model"] == model_label) &
        (variance_df["metric"].isin(bar_metrics))
    ]
    fig_bar.add_trace(go.Bar(
        name=model_label,
        x=sub["metric"],
        y=sub["mean"],
        error_y=dict(type="data", array=sub["std"].tolist(), visible=True),
    ))

fig_bar.update_layout(
    barmode="group",
    title=f"Mean ± std per metric ({N_RUNS} runs, single case)",
    xaxis_title="Metric",
    yaxis_title="Score (0–1)",
    yaxis_range=[0, 1.05],
    legend_title="Model",
)
fig_bar.show()
mlflow.log_figure(fig_bar, "figures/metric_variance_bar.html")


Saved: c:\Users\BRHN\Desktop\SLM-evals\outputs\figures\radar_comparison_test_case_004.html
Saved: c:\Users\BRHN\Desktop\SLM-evals\outputs\figures\radar_comparison_test_case_004.png


## Save reports


In [ ]:
comparison_csv_path = REPORTS_DIR / f"comparison_metrics_{CASE_ID}_nb4.csv"
comparison_json_path = REPORTS_DIR / f"comparison_metrics_{CASE_ID}_nb4.json"
comparison_styled_html_path = REPORTS_DIR / f"comparison_metrics_{CASE_ID}_nb4_styled.html"
comparison_summary_csv_path = REPORTS_DIR / f"comparison_summary_{CASE_ID}_nb4.csv"
judge_csv_path = REPORTS_DIR / f"judge_section_scores_{CASE_ID}_nb4.csv"
judge_styled_html_path = REPORTS_DIR / f"judge_section_scores_{CASE_ID}_nb4_styled.html"
section_winners_csv_path = REPORTS_DIR / f"section_winners_{CASE_ID}_nb4.csv"
weakest_sections_csv_path = REPORTS_DIR / f"weakest_sections_{CASE_ID}_nb4.csv"
ragas_combined_csv_path = REPORTS_DIR / f"ragas_combined_{CASE_ID}_nb4.csv"
ragas_section_combined_csv_path = REPORTS_DIR / f"ragas_section_combined_{CASE_ID}_nb4.csv"
ragas_report_paths = []

comparison_df.to_csv(comparison_csv_path, index=False, encoding="utf-8")
comparison_df.to_json(comparison_json_path, orient="records", force_ascii=False, indent=2)
comparison_styled_html_path.write_text(comparison_styler.to_html(), encoding="utf-8")
comparison_display_df.to_csv(comparison_summary_csv_path, index=False, encoding="utf-8")
judge_df.to_csv(judge_csv_path, index=False, encoding="utf-8")
judge_styled_html_path.write_text(judge_styler.to_html(), encoding="utf-8")
section_winners_df.to_csv(section_winners_csv_path, index=False, encoding="utf-8")
weakest_sections_df.to_csv(weakest_sections_csv_path, index=False, encoding="utf-8")

if "ragas_combined_df" in globals():
    ragas_combined_df.to_csv(ragas_combined_csv_path, index=False, encoding="utf-8")
    ragas_report_paths.append(ragas_combined_csv_path)

if "ragas_section_combined_df" in globals():
    ragas_section_combined_df.to_csv(ragas_section_combined_csv_path, index=False, encoding="utf-8")
    ragas_report_paths.append(ragas_section_combined_csv_path)

print("Saved:", comparison_csv_path)
print("Saved:", comparison_json_path)
print("Saved:", comparison_styled_html_path)
print("Saved:", comparison_summary_csv_path)
print("Saved:", judge_csv_path)
print("Saved:", judge_styled_html_path)
print("Saved:", section_winners_csv_path)
print("Saved:", weakest_sections_csv_path)
for ragas_report_path in ragas_report_paths:
    print("Saved:", ragas_report_path)


Saved: c:\Users\BRHN\Desktop\SLM-evals\outputs\reports\comparison_metrics_test_case_004_nb4.csv
Saved: c:\Users\BRHN\Desktop\SLM-evals\outputs\reports\comparison_metrics_test_case_004_nb4.json
Saved: c:\Users\BRHN\Desktop\SLM-evals\outputs\reports\comparison_metrics_test_case_004_nb4_styled.html
Saved: c:\Users\BRHN\Desktop\SLM-evals\outputs\reports\comparison_summary_test_case_004_nb4.csv
Saved: c:\Users\BRHN\Desktop\SLM-evals\outputs\reports\judge_section_scores_test_case_004_nb4.csv
Saved: c:\Users\BRHN\Desktop\SLM-evals\outputs\reports\judge_section_scores_test_case_004_nb4_styled.html
Saved: c:\Users\BRHN\Desktop\SLM-evals\outputs\reports\section_winners_test_case_004_nb4.csv
Saved: c:\Users\BRHN\Desktop\SLM-evals\outputs\reports\weakest_sections_test_case_004_nb4.csv
Saved: c:\Users\BRHN\Desktop\SLM-evals\outputs\reports\ragas_combined_test_case_004_nb4.csv
Saved: c:\Users\BRHN\Desktop\SLM-evals\outputs\reports\ragas_section_combined_test_case_004_nb4.csv


## MLflow logging and reports

In [ ]:
mlflow.log_params({
    "phase": "evaluation",
    "case_id": CASE_ID,
    "rows_evaluated": len(outputs_df),
    "judge_model": judge_model_name,
    "source_outputs_csv": str(OUTPUTS_CSV_PATH),
    "judge_cache_path": str(JUDGE_CACHE_PATH),
    "force_rerun_judge": FORCE_RERUN_JUDGE,
    "run_ragas": RUN_RAGAS,
    "ragas_runs_per_model": RAGAS_RUNS_PER_MODEL,
    "ragas_total_runs": RAGAS_TOTAL_RUNS,
    "ragas_judge_model": RAGAS_JUDGE_MODEL,
    "score_weights": json.dumps(SCORE_WEIGHTS),
})
mlflow.log_param(
    "eval_scope",
    f"single_case_n_runs_{N_RUNS}_directional_only_scale_to_n20_for_significance",
)

mlflow.log_metrics({
    "best_final_score": float(comparison_df["final_score"].max()),
    "mean_final_score": float(comparison_df["final_score"].mean()),
    "mean_judge_quality_score": float(comparison_df["judge_quality_score"].mean()),
    "mean_safety_score": float(comparison_df["safety_score"].mean()),
    "mean_section_score": float(judge_df["section_score"].mean()),
    "judge_cache_hits": float(cache_hits),
    "judge_cache_misses": float(cache_misses),
})

if use_ensemble_judge and pd.notna(agreement_mean):
    mlflow.log_metric("inter_judge_agreement_mean", float(agreement_mean))

mlflow.log_artifact(str(comparison_csv_path), artifact_path="evaluation_reports")
mlflow.log_artifact(str(comparison_json_path), artifact_path="evaluation_reports")
mlflow.log_artifact(str(comparison_styled_html_path), artifact_path="evaluation_reports")
mlflow.log_artifact(str(comparison_summary_csv_path), artifact_path="evaluation_reports")
mlflow.log_artifact(str(judge_csv_path), artifact_path="evaluation_reports")
mlflow.log_artifact(str(judge_styled_html_path), artifact_path="evaluation_reports")
mlflow.log_artifact(str(section_winners_csv_path), artifact_path="evaluation_reports")
mlflow.log_artifact(str(weakest_sections_csv_path), artifact_path="evaluation_reports")
for ragas_report_path in globals().get("ragas_report_paths", []):
    mlflow.log_artifact(str(ragas_report_path), artifact_path="evaluation_reports")
mlflow.log_artifact(str(JUDGE_CACHE_PATH), artifact_path="evaluation_cache")
mlflow.log_artifact(str(radar_html_path), artifact_path="evaluation_figures")
mlflow.log_artifact(str(radar_png_path), artifact_path="evaluation_figures")

print("MLflow logging done.")


MLflow logging done.


In [ ]:
mlflow.end_run()

🏃 View run nb4_evaluation_test_case_004 at: http://127.0.0.1:5000/#/experiments/4/runs/44c74a49b6694c3aa3dbfaaf107d3926
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4


---
## 📋 Executive summary for supervisor review

In [ ]:
# ── Executive summary — auto-generated from evaluation results ──────────────
# Intended for supervisor review. Summarises key findings in plain English.

winner_label = winner["model_label"]
fastest_label = fastest["model_label"]

print("=" * 70)
print("EVALUATION EXECUTIVE SUMMARY")
print(f"Case: {CASE_ID} | Runs per model: {N_RUNS} | Temperatures: {list(outputs_df['temperature'].unique())}")
print("=" * 70)

print(f"\n{'OVERALL WINNER':─<50}")
print(f"  {winner_label} — Final score: {winner['final_score']:.1%}")
for _, row in comparison_df.sort_values("rank").iterrows():
    print(f"  {'✓' if row['model_label'] == winner_label else '○'} {row['model_label']:12s} "
          f"Final: {row['final_score']:.1%} | "
          f"Judge: {row['judge_quality_score']:.1%} | "
          f"Safety: {row['safety_score']:.1%} | "
          f"Latency: {row['latency_ms']:.0f}ms")

print(f"\n{'METRIC LAYER HIGHLIGHTS':─<50}")
for layer_name, metrics in METRIC_LAYERS.items():
    available = [m for m in metrics if m in variance_df["metric"].values]
    if not available:
        continue
    print(f"\n  {layer_name}")
    for metric in available:
        sub = variance_df[variance_df["metric"] == metric]
        if sub.empty:
            continue
        best_model = sub.loc[sub["mean"].idxmax(), "model"]
        best_mean  = sub["mean"].max()
        best_std   = sub.loc[sub["mean"].idxmax(), "std"]
        print(f"    {metric:<35s} best: {best_model} ({best_mean:.3f} ± {best_std:.3f})")

print(f"\n{'SECTION-LEVEL WEAKNESSES':─<50}")
for model_label, weakest in weakest_sections_df.groupby("model_label"):
    for _, row in weakest.iterrows():
        print(f"  {model_label:12s} #{int(row['weakness_rank'])} weak section: "
              f"{row['section']} ({row['section_score']:.1%})")

print(f"\n{'OPERATIONAL':─<50}")
for _, row in ops_df[ops_df["metric"] == "latency_sec"].iterrows():
    print(f"  {row['model']:12s} latency: {row['mean']:.2f}s ± {row['std']:.2f}s")
for _, row in ops_df[ops_df["metric"] == "total_tokens"].iterrows():
    print(f"  {row['model']:12s} tokens:  {row['mean']:.0f} avg per run")

if use_ensemble_judge and pd.notna(agreement_mean):
    quality = "reliable ✓" if agreement_mean > 0.85 else ("review needed ⚠" if agreement_mean < 0.70 else "acceptable")
    print(f"\n{'JUDGE RELIABILITY':─<50}")
    print(f"  Inter-judge agreement: {agreement_mean:.3f} — {quality}")

print(f"\n{'INTERPRETATION NOTE':─<50}")
print(f"  This is a single-case directional benchmark (n=1 test case, {N_RUNS} temperature runs).")
print(f"  Scores indicate direction of advantage only.")
print(f"  Statistical significance requires n≥20 diverse test cases.")
print("=" * 70)